# EDA — base de treino e teste (Fake.br + FakeRecogna + Boatos.org)

Análise da base que substitui o Fakepedia (ver `doc/eda.md` para o porquê).

| `base` | O que é | Rótulo vem de | Split |
|---|---|---|---|
| `fakebr` | Fake.br-Corpus, 2016–2018 | curadoria do NILC | treino |
| `fakerecogna` | URLs do FakeRecogna (2019–2021) com o texto **recoletado**: boatos do Boatos.org e notícias de G1/UOL/Extra | agência de checagem / portal | treino |
| `boatos_recente` | checagens do Boatos.org de 2024 em diante | agência de checagem | teste |
| `factcheck` | Google Fact Check Tools API (quando houver chave) | agências de checagem | treino e teste |

Para gerar os dados: `bash scripts/baixar_dados.sh`, os coletores em `scripts/` e
`python scripts/preparar_dados.py`. Cada coluna `fonte` guarda o domínio de onde veio o
texto, para medir o atalho de fonte.

In [ ]:
import json
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = RAIZ / "data" / "raw"

CORES = {"fake": "#eb6834", "true": "#2a78d6"}
TINTA, TINTA_2 = "#0b0b0b", "#52514e"
ORDEM_BASES = ["fakebr", "fakerecogna", "boatos_recente", "factcheck"]
plt.rcParams.update({
    "figure.dpi": 110, "figure.figsize": (9, 4), "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e5e0",
    "grid.linewidth": 0.8, "axes.axisbelow": True, "axes.edgecolor": "#b8b7b0",
    "axes.labelcolor": TINTA_2, "xtick.color": TINTA_2, "ytick.color": TINTA_2,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False,
})
pd.set_option("display.max_colwidth", 110)

df = pd.read_json(RAIZ / "data" / "processed" / "dataset.jsonl", lines=True, dtype={"par_id": "Int64"})
df["data"] = pd.to_datetime(df["data"], errors="coerce")
df["n_palavras"] = df.texto.str.split().str.len()
bases = [b for b in ORDEM_BASES if b in set(df.base)]
visao = df.groupby(["base", "split", "rotulo"]).size().unstack(fill_value=0).reindex(bases, level=0)
visao["total"] = visao.sum(axis=1)
visao

## 1. Por que o FakeRecogna não entra como vem

O CSV publicado traz o texto já **lematizado e com parte das stopwords removidas**, e a
classe *fake* é, em grande parte, o texto da agência de checagem descrevendo o boato,
não o boato em si.

In [ ]:
fr = pd.read_csv(RAW / "FakeRecogna.csv")
fr["rotulo"] = fr.Classe.map({0: "fake", 1: "true"})
fr["fonte"] = fr.URL.str.extract(r"https?://(?:www\d?\.)?([^/]+)")[0]
fr["Noticia"] = fr.Noticia.fillna("").str.lower()
print("Exemplo do texto como vem no CSV:\n ", fr.Noticia.iloc[1][:220])

In [ ]:
# termo no CSV lematizado -> radical equivalente no texto original
TERMOS_CHECADOR = {"circular": "circul", "rede social": "redes sociais", "falso": "fals", "fake": "fake",
                   "whatsapp": "whatsapp", "checagem": "checage", "enganoso": "engano", "montagem": "montage"}
def pct_com(textos, termo):
    return 100 * textos.str.lower().str.contains(termo, regex=False).mean()

checador_csv = pd.DataFrame({t: fr.groupby("rotulo").Noticia.apply(pct_com, termo=t) for t in TERMOS_CHECADOR}).T
subtitulo_boato = fr.groupby("rotulo").Subtitulo.apply(lambda s: 100 * s.fillna("").str.startswith("Boato").mean())
print("% com subtítulo começando por 'Boato –':", subtitulo_boato.round(1).to_dict())
checador_csv.round(1).rename(columns=lambda c: f"% {c}")

Termos típicos de quem **escreve a checagem** ("circula nas redes sociais", "é falso",
"montagem") aparecem muito mais nas falsas. Um modelo treinado nesse texto aprende a
reconhecer checagens, e não boatos. Por isso usamos só a URL e o rótulo do FakeRecogna e
recoletamos o texto (seção 2); a seção 8 confere se esse vocabulário sumiu.

## 2. Cobertura da recoleta

In [ ]:
def ler_jsonl(caminho):
    return pd.DataFrame([json.loads(l) for l in caminho.read_text(encoding="utf-8").splitlines()]) \
        if caminho.exists() else pd.DataFrame()

boatos_raw = ler_jsonl(RAW / "boatos" / "boatos.jsonl")
noticias_raw = ler_jsonl(RAW / "noticias" / "noticias.jsonl")

cobertura = fr.groupby(["rotulo", "fonte"]).size().rename("no FakeRecogna").to_frame()
recoletadas = pd.concat([
    boatos_raw[boatos_raw.origem == "fakerecogna"].assign(rotulo="fake", fonte="boatos.org",
        ok=lambda d: (d.status == 200) & (d.texto_boato.notna() | d.alegacao.notna())),
    noticias_raw.assign(rotulo="true", fonte=lambda d: d.url.str.extract(r"https?://(?:www\d?\.)?([^/]+)")[0],
        ok=lambda d: (d.status == 200) & d.texto.notna()),
])
cobertura["recoletadas com texto"] = recoletadas[recoletadas.ok].groupby(["rotulo", "fonte"]).size()
cobertura = cobertura.fillna(0).astype(int)
cobertura["%"] = (100 * cobertura["recoletadas com texto"] / cobertura["no FakeRecogna"]).round(0)
cobertura.sort_values("no FakeRecogna", ascending=False).groupby(level=0).head(7)

In [ ]:
resumo = cobertura.groupby(level=0)[["no FakeRecogna", "recoletadas com texto"]].sum()
resumo["%"] = (100 * resumo["recoletadas com texto"] / resumo["no FakeRecogna"]).round(0)
print("status HTTP das notícias verdadeiras:", noticias_raw.status.value_counts().to_dict())
resumo

- **Falsas:** só as 2.484 do Boatos.org têm extrator por enquanto. As das outras agências
  (Fato ou Fake, e-farsas, AFP, Comprova, UOL Confere) ficam de fora até termos um
  coletor para cada uma, ou até a Fact Check API cobrir esse papel.
- **Verdadeiras:** 63% das URLs são do `noticias.uol.com.br` e do `economia.uol.com.br`,
  que bloqueiam qualquer acesso automatizado (403, inclusive no `robots.txt`). Não
  tentamos contornar. Das demais, algumas já saíram do ar (404).

## 3. Qualidade dos textos

In [ ]:
df["n_car"] = df.texto.str.len()
df["curto (<20 palavras)"] = df.n_palavras < 20
df["texto repetido"] = df.duplicated("texto", keep=False)
df["início repetido (200 car.)"] = df.texto.str[:200].duplicated(keep=False)
df.groupby(["base", "rotulo"])[["curto (<20 palavras)", "texto repetido", "início repetido (200 car.)"]] \
    .mean().mul(100).round(1).reindex(bases, level=0)

In [ ]:
# Repetições entre treino e teste: o Boatos.org republica boatos antigos.
treino_inicios = set(df[df.split == "treino"].texto.str[:200])
teste = df[df.split == "teste"]
print(f"{teste.texto.str[:200].isin(treino_inicios).sum()} textos do teste começam igual a um texto do treino")

In [ ]:
if not boatos_raw.empty:
    b = boatos_raw[boatos_raw.status == 200]
    origem_texto = pd.DataFrame({
        "tem texto citado": b.texto_boato.notna(),
        "só a alegação": b.texto_boato.isna() & b.alegacao.notna(),
        "nada extraído": b.texto_boato.isna() & b.alegacao.isna(),
    }).groupby(b.origem.where(b.origem == "fakerecogna", "sitemap 2024–2026")).mean().mul(100).round(1)
    display(origem_texto)

No Boatos.org, o texto preferido é a **mensagem citada** (o boato como circulou). Quando
a página não cita a mensagem, porque o boato era só imagem ou vídeo, usamos a alegação
"Boato – ...", que resume o boato na voz dele.

## 4. Período

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
com_data = df.dropna(subset=["data"])
com_data = com_data[com_data.data >= "2015-01-01"]
for (base, rotulo), g in com_data.groupby(["base", "rotulo"]):
    serie = g.set_index("data").resample("QS").size()
    estilo = "-" if base != "boatos_recente" else "--"
    ax.plot(serie.index, serie.values, estilo, color=CORES[rotulo], lw=2, alpha=0.9 if base != "fakebr" else 0.55,
            label=f"{base} · {rotulo}")
ax.set_title("Notícias por trimestre de publicação")
ax.set_ylabel("notícias")
ax.legend(ncol=3, fontsize=8, loc="upper left")
df.groupby(["base", "rotulo"]).data.agg(["min", "median", "max"]).reindex(bases, level=0)

As três fontes cobrem **épocas que não se sobrepõem**: Fake.br até 2018, FakeRecogna em
2019–2021 (auge da Covid) e o teste de 2024 em diante. É isso que torna o teste temporal:
o modelo vai ser avaliado em assuntos que não existiam no treino. A contrapartida é que
a data (e o assunto da época) identifica a base, então ela nunca entra como feature.

## 5. Fontes: o atalho continua

In [ ]:
fontes = (df.groupby(["base", "rotulo"]).fonte.value_counts(normalize=True).mul(100).round(1)
          .groupby(level=[0, 1]).head(4).rename("% da classe").to_frame())
fontes.reindex(bases, level=0)

Em todas as bases, **cada classe vem de um conjunto de sites diferente**. No
FakeRecogna reconstruído, 100% das falsas vêm do Boatos.org e as verdadeiras vêm de
portais. A consequência vai além da fonte: é também o **gênero do texto**, como mostram as
duas próximas seções.

## 6. Tamanho

In [ ]:
fig, axes = plt.subplots(1, len(bases), figsize=(4 * len(bases), 3.6), sharey=False)
bins = np.logspace(0.5, 4, 40)
for ax, base in zip(np.atleast_1d(axes), bases):
    g = df[df.base == base]
    for rotulo in ["true", "fake"]:
        s = g[g.rotulo == rotulo].n_palavras
        if len(s):
            ax.hist(s, bins=bins, color=CORES[rotulo], alpha=0.6, label=rotulo)
    ax.set_xscale("log")
    ax.set_title(base)
    ax.set_xlabel("palavras")
np.atleast_1d(axes)[0].set_ylabel("textos")
np.atleast_1d(axes)[0].legend()
fig.tight_layout()
df.groupby(["base", "rotulo"]).n_palavras.describe()[["count", "25%", "50%", "75%"]].round(0).reindex(bases, level=0)

## 7. Estilo: mensagem de rede social × matéria

In [ ]:
# Imperativos que pedem para passar a mensagem adiante ("divulgou", "encaminhado" não contam).
PEDE_COMPARTILHAR = re.compile(r"(compartilhe[m]?|repasse[m]?|divulgue[m]?|encaminhe[m]?|espalhe[m]?|"
                               r"envie para|mande para|passe adiante)", re.I)
EMOJI = re.compile("[\U0001F300-\U0001FAFF\u2600-\u27BF]")
def estilo(texto):
    palavras = re.findall(r"\w+", texto)
    n = max(len(palavras), 1)
    return pd.Series({
        "% palavras em CAIXA ALTA": 100 * sum(p.isupper() and len(p) > 1 for p in palavras) / n,
        "! por 100 palavras": 100 * texto.count("!") / n,
        "tem emoji": bool(EMOJI.search(texto)),
        "pede para compartilhar": bool(PEDE_COMPARTILHAR.search(texto)),
        "tem URL": bool(re.search(r"https?://|www\.", texto)),
    })
estilos = df.join(df.texto.apply(estilo))
tabela_estilo = estilos.groupby(["base", "rotulo"])[[
    "% palavras em CAIXA ALTA", "! por 100 palavras", "tem emoji", "pede para compartilhar", "tem URL"
]].mean()
for c in ["tem emoji", "pede para compartilhar", "tem URL"]:
    tabela_estilo[c] *= 100
tabela_estilo.round(2).reindex(bases, level=0)

Nas bases que usam o Boatos.org, as falsas são **mensagens de rede social** (CAIXA ALTA,
exclamações, emoji, "compartilhe!") e as verdadeiras são **matérias jornalísticas**. Um
classificador aprende essa diferença de gênero com facilidade, e ela explica boa parte de
qualquer acurácia alta. Para o rótulo depender da veracidade e não do gênero, a classe
verdadeira precisa de **alegações que circularam e foram confirmadas**. Esse é o papel da
Fact Check API.

## 8. A recoleta tirou o vocabulário do checador?

In [ ]:
fr_rec = df[df.base == "fakerecogna"]
depois = pd.DataFrame({t: fr_rec.groupby("rotulo").texto.apply(pct_com, termo=r) for t, r in TERMOS_CHECADOR.items()}).T
pd.concat({"CSV original (%)": checador_csv, "recoletado (%)": depois}, axis=1).round(1)

Como o CSV é lematizado, a busca no texto recoletado usa o radical equivalente
("circular" → "circul", que pega "circula", "circulando"...). Os termos que sobram nas falsas
recoletadas vêm, em geral, do próprio boato ("é fake", "a imprensa não mostra", "vídeo").

## 9. Vocabulário característico (FakeRecogna recoletado)

In [ ]:
TOKEN = re.compile(r"[a-záàâãéêíóôõúüç]+")
def contar(textos):
    c = Counter()
    for t in textos:
        c.update(TOKEN.findall(t.lower()))
    return c

def log_odds(ca, cb, alpha=0.01, min_freq=20):
    fundo = ca + cb
    a0 = alpha * sum(fundo.values())
    na, nb = sum(ca.values()), sum(cb.values())
    linhas = []
    for w, f in fundo.items():
        if f < min_freq:
            continue
        aw = alpha * f
        ya, yb = ca[w], cb[w]
        delta = np.log((ya + aw) / (na + a0 - ya - aw)) - np.log((yb + aw) / (nb + a0 - yb - aw))
        linhas.append((w, delta / np.sqrt(1 / (ya + aw) + 1 / (yb + aw))))
    return pd.DataFrame(linhas, columns=["palavra", "z"]).set_index("palavra")

lo = log_odds(contar(fr_rec[fr_rec.rotulo == "fake"].texto), contar(fr_rec[fr_rec.rotulo == "true"].texto))
pd.concat({
    "mais típicas de FAKE": lo.nlargest(20, "z").reset_index(),
    "mais típicas de TRUE": lo.nsmallest(20, "z").reset_index(),
}, axis=1).round(1)

## 10. O teste temporal traz assuntos novos

In [ ]:
treino_vocab = contar(df[df.split == "treino"].texto)
teste_df = df[df.split == "teste"]
if len(teste_df):
    teste_vocab = contar(teste_df.texto)
    novas = pd.Series({w: n for w, n in teste_vocab.items() if treino_vocab[w] == 0 and n >= 8}).sort_values(ascending=False)
    print(f"{len(novas)} palavras aparecem ≥ 8 vezes no teste e nunca no treino. As mais frequentes:")
    print(", ".join(novas.head(40).index))
    cobertos = sum(n for w, n in teste_vocab.items() if treino_vocab[w] > 0) / sum(teste_vocab.values())
    print(f"\n{100 * cobertos:.1f}% das ocorrências de palavras do teste existem no vocabulário do treino")

In [ ]:
if len(teste_df):
    cat = pd.concat({
        "treino (fakerecogna, fake)": df[(df.base == "fakerecogna") & (df.rotulo == "fake")].categoria.value_counts(normalize=True),
        "teste (boatos_recente)": teste_df.categoria.value_counts(normalize=True),
    }, axis=1).fillna(0).mul(100).round(1)
    display(cat.sort_values("teste (boatos_recente)", ascending=False))

## 11. Fact Check API

In [ ]:
fc = ler_jsonl(RAW / "factcheck" / "checagens.jsonl")
if fc.empty:
    print("Ainda sem dados: rode scripts/coletar_factcheck.py com FACTCHECK_API_KEY no .env.")
else:
    fc["ano"] = fc.data_checagem.str[:4]
    display(fc.veredito.str.lower().str.strip().value_counts().head(25).to_frame("checagens"))
    display(fc.groupby(["site_agencia", "ano"]).size().unstack(fill_value=0))

## 12. Conclusões

_(preenchidas após rodar com os dados completos)_